In [1]:
%pip install -U sentence-transformers chromadb

  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl.metadata (21 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------------------- 740.6/740.6 kB 6.1 MB/s  0:00:00
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ---------------------------------------

In [1]:
import sentence_transformers
import chromadb

print("Sentence Transformers version:", sentence_transformers.__version__)
print("ChromaDB version:", chromadb.__version__)
print("Packages imported successfully.")

Sentence Transformers version: 6.1.0
ChromaDB version: 1.5.9
Packages imported successfully.


In [2]:
from pathlib import Path
import json

project_root = Path.cwd().parent
chunks_file = project_root / "data" / "processed" / "knowledge_base_chunks.json"

with chunks_file.open("r", encoding="utf-8") as file:
    chunks = json.load(file)

print(f"Loaded {len(chunks)} chunks.")
print("\nFirst chunk:")
print(chunks[0])

Loaded 27 chunks.

First chunk:
{'content': '# NovaMart Australia — Account and Privacy Support\n\n## Creating an account\n\nCustomers may create a NovaMart account using their name, email address and a secure password.\n\nAn account is optional. Customers may also place orders using guest checkout.', 'source': 'account_privacy.md', 'chunk_id': 'account_privacy.md_1'}


In [3]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\arunp\anaconda3\envs\novamart-rag\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arunp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [4]:
sample_text = "How can I return a damaged product?"

sample_embedding = embedding_model.encode(
    sample_text,
    normalize_embeddings=True
)

print("Original text:")
print(sample_text)

print("\nEmbedding dimensions:")
print(sample_embedding.shape)

print("\nFirst 10 numbers:")
print(sample_embedding[:10])

Original text:
How can I return a damaged product?

Embedding dimensions:
(384,)

First 10 numbers:
[-0.03275979  0.01538341  0.08718555 -0.03322036  0.0065842  -0.00326712
 -0.00906184  0.05999554 -0.04502528 -0.04812332]


In [5]:
chunk_texts = [chunk["content"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Number of chunks:", len(chunk_texts))
print("Embedding matrix shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of chunks: 27
Embedding matrix shape: (27, 384)


In [6]:
import chromadb

chroma_path = project_root / "data" / "chroma_db"

chroma_client = chromadb.PersistentClient(path=str(chroma_path))

collection = chroma_client.get_or_create_collection(
    name="novamart_support",
    metadata={"hnsw:space": "cosine"}
)

chunk_ids = [chunk["chunk_id"] for chunk in chunks]
chunk_metadata = [{"source": chunk["source"]} for chunk in chunks]

collection.upsert(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadata
)

print(f"Stored chunks in ChromaDB: {collection.count()}")
print(f"Database location: {chroma_path}")

Stored chunks in ChromaDB: 27
Database location: C:\Users\arunp\Desktop\rag-assistant\data\chroma_db


In [7]:
customer_question = "My order arrived damaged. What should I do?"

question_embedding = embedding_model.encode(
    customer_question,
    normalize_embeddings=True
)

results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print("Customer question:")
print(customer_question)

print("\nTop retrieved chunks:")

for position in range(3):
    source = results["metadatas"][0][position]["source"]
    document = results["documents"][0][position]
    distance = results["distances"][0][position]
    similarity = 1 - distance

    print(f"\n--- Result {position + 1} ---")
    print(f"Source: {source}")
    print(f"Similarity score: {similarity:.3f}")
    print(document)

Customer question:
My order arrived damaged. What should I do?

Top retrieved chunks:

--- Result 1 ---
Source: returns_refunds.md
Similarity score: 0.496
# NovaMart Australia — Returns and Refunds Policy

## Damaged or incorrect products

Customers must contact NovaMart Support within 7 calendar days of delivery if a product arrives damaged, defective or incorrect.

NovaMart will provide a prepaid return label. The customer may choose:

- A replacement
- A repair, where appropriate
- A full refund to the original payment method

Customers should provide their order number and clear photos of the product and packaging.

--- Result 2 ---
Source: product_warranty.md
Similarity score: 0.421
# NovaMart Australia — Product Warranty Policy

## Making a warranty claim

Customers should contact NovaMart Support and provide:

- Their order number
- A description of the problem
- Photos or a short video showing the fault
- The product serial number, when applicable

NovaMart may offer a repair, 

In [8]:
def retrieve_chunks(question, number_of_results=3):
    question_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=number_of_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    for position in range(number_of_results):
        retrieved_chunks.append({
            "content": results["documents"][0][position],
            "source": results["metadatas"][0][position]["source"],
            "similarity": 1 - results["distances"][0][position]
        })

    return retrieved_chunks

In [9]:
test_question = "Can I return an unused item after 30 days?"
retrieved_chunks = retrieve_chunks(test_question)

print("Question:", test_question)

for position, chunk in enumerate(retrieved_chunks, start=1):
    print(f"\n--- Result {position} ---")
    print("Source:", chunk["source"])
    print(f"Similarity: {chunk['similarity']:.3f}")
    print(chunk["content"])

Question: Can I return an unused item after 30 days?

--- Result 1 ---
Source: returns_refunds.md
Similarity: 0.578
# NovaMart Australia — Returns and Refunds Policy

## Change-of-mind returns

Customers may return an unused product within 30 calendar days of delivery.

The product must:

- Be unused and in its original condition
- Include the original packaging, accessories and manuals
- Include proof of purchase

Change-of-mind returns are not accepted after 30 days.

Customers are responsible for return shipping costs for change-of-mind returns.

--- Result 2 ---
Source: returns_refunds.md
Similarity: 0.420
# NovaMart Australia — Returns and Refunds Policy

## Damaged or incorrect products

Customers must contact NovaMart Support within 7 calendar days of delivery if a product arrives damaged, defective or incorrect.

NovaMart will provide a prepaid return label. The customer may choose:

- A replacement
- A repair, where appropriate
- A full refund to the original payment method

C

In [10]:
from getpass import getpass

api_key = getpass("Paste your OpenAI API key here: ")

env_file = project_root / ".env"
env_file.write_text(
    f"OPENAI_API_KEY={api_key}\n",
    encoding="utf-8"
)

gitignore_file = project_root / ".gitignore"

if gitignore_file.exists():
    existing_rules = gitignore_file.read_text(encoding="utf-8")

    if ".env" not in existing_rules.splitlines():
        with gitignore_file.open("a", encoding="utf-8") as file:
            file.write("\n.env\n")
else:
    gitignore_file.write_text(".env\n", encoding="utf-8")

del api_key

print("API key saved in .env")
print(".env added to .gitignore")

Paste your OpenAI API key here:  ········


API key saved in .env
.env added to .gitignore
